In [9]:
import os
import shutil
import pandas as pd

# Set iteration of study and source path
study = 'Study4.0'
condition = 'Predict'
source_dir = "/Users/sm6511/Downloads/raw_pre_3/"

dest_dir = (
    f"/Users/sm6511/Desktop/Prediction-Accomodation-Exp/"
    f"data/{study}/{condition}"
)

target_dates = [
    "2026-05-18",
    "2026-05-19",
    "2026-05-20",
]

DELETE_FAILED_ATTENTION = True

completion_col = "button_end.numClicks"

if condition.lower() == "predict":
    attention_col = "answer_3_right.numClicks"
elif condition.lower() == "accommodate":
    attention_col = "button_3_correct.numClicks"
else:
    raise ValueError("condition must be 'Predict' or 'Accommodate'")

os.makedirs(dest_dir, exist_ok=True)

moved = []
skipped = []
deleted_failed_attention = []

# collect files sorted by earlier date
candidate_files = sorted([
    fname for fname in os.listdir(source_dir)
    if fname.endswith(".csv") and any(d in fname for d in target_dates)
])

# STEP 1: delete failed attention checks
if DELETE_FAILED_ATTENTION:
    for fname in candidate_files:
        src_path = os.path.join(source_dir, fname)

        try:
            df = pd.read_csv(src_path)
            df.columns = df.columns.str.strip()
        except Exception as e:
            print(f"Could not read {fname}: {e}")
            skipped.append((fname, "read error during attention check"))
            continue

        if attention_col not in df.columns:
            print(f"Missing {attention_col} in {fname}")
            skipped.append((fname, "missing attention column"))
            continue

        passed_attention = (
            pd.to_numeric(df[attention_col], errors="coerce")
            .eq(1)
            .any()
        )

        if not passed_attention:
            print(f"🗑️ DELETING FAILED ATTENTION CHECK: {fname}")
            os.remove(src_path)
            deleted_failed_attention.append(fname)

# refresh list of files
candidate_files = sorted([
    fname for fname in os.listdir(source_dir)
    if fname.endswith(".csv") and any(d in fname for d in target_dates)
])

# STEP 2: move completed files, earliest first
for fname in candidate_files:
    src_path = os.path.join(source_dir, fname)

    try:
        df = pd.read_csv(src_path)
        df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"Could not read {fname}: {e}")
        skipped.append((fname, "read error"))
        continue

    if completion_col not in df.columns:
        print(f"Missing {completion_col} in {fname}")
        skipped.append((fname, "missing completion column"))
        continue

    is_complete = (
        pd.to_numeric(df[completion_col], errors="coerce")
        .eq(1)
        .any()
    )

    if is_complete:
        dest_path = os.path.join(dest_dir, fname)
        shutil.move(src_path, dest_path)
        moved.append(fname)
        print(f"MOVED: {fname}")
    else:
        skipped.append((fname, f"{completion_col} != 1"))

print("\n===== SUMMARY =====")

print(f"Deleted failed attention files ({len(deleted_failed_attention)}):")
for f in deleted_failed_attention:
    print(f"  {f}")

print(f"\nMoved files ({len(moved)}):")
for f in moved:
    print(f"  {f}")

print(f"\nSkipped files ({len(skipped)}):")
for f, reason in skipped:
    print(f"  {f} — {reason}")

Missing answer_3_right.numClicks in 008_test_2026-05-18_10h52.32.174.csv
🗑️ DELETING FAILED ATTENTION CHECK: 018_test_2026-05-18_17h23.41.597.csv
🗑️ DELETING FAILED ATTENTION CHECK: 032_test_2026-05-18_13h51.48.781.csv
🗑️ DELETING FAILED ATTENTION CHECK: 034_test_2026-05-18_12h47.28.002.csv
🗑️ DELETING FAILED ATTENTION CHECK: 049_test_2026-05-18_13h47.19.890.csv
Missing answer_3_right.numClicks in 051_test_2026-05-18_12h49.07.078.csv
Missing answer_3_right.numClicks in 053_test_2026-05-18_13h50.10.766.csv
Could not read 054_test_2026-05-18_11h00.45.684.csv: No columns to parse from file
🗑️ DELETING FAILED ATTENTION CHECK: 059_test_2026-05-18_17h22.57.613.csv
🗑️ DELETING FAILED ATTENTION CHECK: 066_test_2026-05-18_13h50.01.855.csv
🗑️ DELETING FAILED ATTENTION CHECK: 067_test_2026-05-18_12h47.35.537.csv
🗑️ DELETING FAILED ATTENTION CHECK: 079_test_2026-05-18_14h26.17.069.csv
🗑️ DELETING FAILED ATTENTION CHECK: 079_test_2026-05-19_09h38.07.406.csv
🗑️ DELETING FAILED ATTENTION CHECK: 083_t